In [1]:
# =========================
# GoEmotions EXP1: RoBERTa Baseline
# =========================

import random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "roberta-base"
MAX_LEN = 128
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 4
THRESH = 0.5

print("Loading GoEmotions (simplified)...")
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
emotion_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(emotion_names)

print(f"Sizes: train={len(ds['train'])}, val={len(ds['validation'])}, test={len(ds['test'])}")
print(f"Num labels: {NUM_LABELS}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GoEmotionsDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=128, num_labels=28):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.num_labels = num_labels

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        y = torch.zeros(self.num_labels, dtype=torch.float32)
        for lab in item["labels"]:
            y[lab] = 1.0

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": y,
        }

class RobertaBaseline(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # <s> token
        return self.classifier(cls)

def evaluate(model, loader, device, thresh=0.5, return_arrays=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].cpu().numpy()

            logits = model(ids, mask)
            probs = torch.sigmoid(logits).cpu().numpy()

            all_probs.append(probs)
            all_labels.append(y)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds = (all_probs > thresh).astype(int)

    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    micro = f1_score(all_labels, preds, average="micro", zero_division=0)

    if return_arrays:
        return all_probs, all_labels, macro, micro
    return macro, micro

def emotion_level_report(probs, labels, emotion_names, threshold=0.5, sort_by="f1"):
    preds = (probs > threshold).astype(int)
    p, r, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    df = pd.DataFrame({"emotion": emotion_names, "precision": p, "recall": r, "f1": f1, "support": support})
    df["pred_count"] = preds.sum(axis=0)
    df["true_count"] = labels.sum(axis=0)
    df["pred_rate"] = (df["pred_count"] / max(len(preds), 1)).round(4)
    return df.sort_values(sort_by, ascending=False).reset_index(drop=True)

train_ds = GoEmotionsDataset(ds["train"], tokenizer, MAX_LEN, NUM_LABELS)
val_ds   = GoEmotionsDataset(ds["validation"], tokenizer, MAX_LEN, NUM_LABELS)
test_ds  = GoEmotionsDataset(ds["test"], tokenizer, MAX_LEN, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

model = RobertaBaseline(MODEL_NAME, NUM_LABELS).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val = -1.0
best_path = "roberta_exp1_best.pt"

print("\nTraining EXP1...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_macro, val_micro = evaluate(model, val_loader, DEVICE, THRESH, return_arrays=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | val_macro={val_macro:.4f} | val_micro={val_micro:.4f}")

    if val_macro > best_val:
        best_val = val_macro
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ saved best (val_macro={best_val:.4f})")

print("\nTesting best checkpoint...")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_probs, test_labels, test_macro, test_micro = evaluate(model, test_loader, DEVICE, THRESH, return_arrays=True)

per_class_df = emotion_level_report(test_probs, test_labels, emotion_names, threshold=THRESH, sort_by="f1")
display(per_class_df.head(10))
display(per_class_df.tail(10))

print(f"\nTEST @0.5 | Macro-F1={test_macro:.4f} | Micro-F1={test_micro:.4f}")
print("Done.")


/home/jovyan/hf-goemotions/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GoEmotions (simplified)...
Sizes: train=43410, val=5426, test=5427
Num labels: 28


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f48daaa5-6f37-4f7a-af74-8bf85a707109)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/added_tokens.json
Retrying in 1s [Retry 1/5].



Training EXP1...
Epoch 1/4 | loss=0.1250 | val_macro=0.3743 | val_micro=0.5250
  ✓ saved best (val_macro=0.3743)
Epoch 2/4 | loss=0.0809 | val_macro=0.4544 | val_micro=0.5895
  ✓ saved best (val_macro=0.4544)
Epoch 3/4 | loss=0.0682 | val_macro=0.4919 | val_micro=0.5878
  ✓ saved best (val_macro=0.4919)
Epoch 4/4 | loss=0.0553 | val_macro=0.5015 | val_micro=0.5860
  ✓ saved best (val_macro=0.5015)

Testing best checkpoint...


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
0,gratitude,0.938053,0.903409,0.920405,352,339,352.0,0.0625
1,love,0.774436,0.865546,0.817460,238,266,238.0,0.0490
2,amusement,0.770270,0.863636,0.814286,264,296,264.0,0.0545
3,admiration,0.669691,0.732143,0.699526,504,551,504.0,0.1015
4,fear,0.666667,0.692308,0.679245,78,81,78.0,0.0149
5,remorse,0.605634,0.767857,0.677165,56,71,56.0,0.0131
6,neutral,0.728603,0.557359,0.631579,1787,1367,1787.0,0.2519
7,joy,0.655172,0.590062,0.620915,161,145,161.0,0.0267
8,sadness,0.614286,0.551282,0.581081,156,140,156.0,0.0258
9,optimism,0.622517,0.505376,0.557864,186,151,186.0,0.0278


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
18,embarrassment,0.583333,0.378378,0.459016,37,24,37.0,0.0044
19,caring,0.504762,0.392593,0.441667,135,105,135.0,0.0193
20,relief,1.000000,0.272727,0.428571,11,3,11.0,0.0006
21,approval,0.480287,0.381766,0.425397,351,279,351.0,0.0514
22,confusion,0.451613,0.366013,0.404332,153,124,153.0,0.0228
23,disapproval,0.458128,0.348315,0.395745,267,203,267.0,0.0374
24,annoyance,0.457286,0.284375,0.350674,320,199,320.0,0.0367
25,disappointment,0.423077,0.218543,0.288210,151,78,151.0,0.0144
26,realization,0.367647,0.172414,0.234742,145,68,145.0,0.0125
27,grief,0.000000,0.000000,0.000000,6,0,6.0,0.0000



TEST @0.5 | Macro-F1=0.5157 | Micro-F1=0.5918
Done.


In [2]:
# =========================
# GoEmotions EXP2: RoBERTa + pos_weight
# =========================

import random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "roberta-base"
MAX_LEN = 128
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 4
THRESH = 0.5

print("Loading GoEmotions (simplified)...")
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
emotion_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(emotion_names)
print(f"Sizes: train={len(ds['train'])}, val={len(ds['validation'])}, test={len(ds['test'])}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GoEmotionsDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=128, num_labels=28):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.num_labels = num_labels
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        enc = self.tokenizer(text, max_length=self.max_len, truncation=True,
                             padding="max_length", return_tensors="pt")
        y = torch.zeros(self.num_labels, dtype=torch.float32)
        for lab in item["labels"]:
            y[lab] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": y,
        }

class RobertaBaseline(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

def compute_pos_weights(hf_train_split, num_labels, device):
    pos = np.zeros(num_labels, dtype=np.float64)
    for ex in hf_train_split:
        for lab in ex["labels"]:
            pos[lab] += 1.0
    N = len(hf_train_split)
    neg = N - pos
    pos = np.clip(pos, 1.0, None)
    pos_w = neg / pos
    pos_w = np.clip(pos_w, 1.0, 50.0)
    return torch.tensor(pos_w, dtype=torch.float32, device=device)

def evaluate(model, loader, device, thresh=0.5, return_arrays=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["labels"].cpu().numpy()

            logits = model(ids, mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs); all_labels.append(y)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds = (all_probs > thresh).astype(int)

    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    micro = f1_score(all_labels, preds, average="micro", zero_division=0)
    if return_arrays:
        return all_probs, all_labels, macro, micro
    return macro, micro

def emotion_level_report(probs, labels, emotion_names, threshold=0.5, sort_by="f1"):
    preds = (probs > threshold).astype(int)
    p, r, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    df = pd.DataFrame({"emotion": emotion_names, "precision": p, "recall": r, "f1": f1, "support": support})
    df["pred_count"] = preds.sum(axis=0)
    df["true_count"] = labels.sum(axis=0)
    df["pred_rate"] = (df["pred_count"] / max(len(preds), 1)).round(4)
    return df.sort_values(sort_by, ascending=False).reset_index(drop=True)

train_ds = GoEmotionsDataset(ds["train"], tokenizer, MAX_LEN, NUM_LABELS)
val_ds   = GoEmotionsDataset(ds["validation"], tokenizer, MAX_LEN, NUM_LABELS)
test_ds  = GoEmotionsDataset(ds["test"], tokenizer, MAX_LEN, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False)

model = RobertaBaseline(MODEL_NAME, NUM_LABELS).to(DEVICE)

pos_wts = compute_pos_weights(ds["train"], NUM_LABELS, DEVICE)
print(f"[EXP2] pos_weight range: {pos_wts.min().item():.2f} - {pos_wts.max().item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_wts)
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val = -1.0
best_path = "roberta_exp2_posweight_best.pt"

print("\nTraining EXP2...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, mask)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_macro, val_micro = evaluate(model, val_loader, DEVICE, THRESH, return_arrays=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | val_macro={val_macro:.4f} | val_micro={val_micro:.4f}")

    if val_macro > best_val:
        best_val = val_macro
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ saved best (val_macro={best_val:.4f})")

print("\nTesting best checkpoint...")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_probs, test_labels, test_macro, test_micro = evaluate(model, test_loader, DEVICE, THRESH, return_arrays=True)

per_class_df = emotion_level_report(test_probs, test_labels, emotion_names, threshold=THRESH, sort_by="f1")
display(per_class_df.head(10))
display(per_class_df.tail(10))

print(f"\nTEST @0.5 | Macro-F1={test_macro:.4f} | Micro-F1={test_micro:.4f}")
print("Done.")


Loading GoEmotions (simplified)...
Sizes: train=43410, val=5426, test=5427
[EXP2] pos_weight range: 2.05 - 50.00

Training EXP2...
Epoch 1/4 | loss=0.6944 | val_macro=0.3484 | val_micro=0.3641
  ✓ saved best (val_macro=0.3484)
Epoch 2/4 | loss=0.4907 | val_macro=0.3925 | val_micro=0.4315
  ✓ saved best (val_macro=0.3925)
Epoch 3/4 | loss=0.3741 | val_macro=0.4106 | val_micro=0.4453
  ✓ saved best (val_macro=0.4106)
Epoch 4/4 | loss=0.2819 | val_macro=0.4455 | val_micro=0.4892
  ✓ saved best (val_macro=0.4455)

Testing best checkpoint...


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
0,gratitude,0.700611,0.977273,0.816133,352,491,352.0,0.0905
1,amusement,0.651948,0.950758,0.773498,264,385,264.0,0.0709
2,love,0.623269,0.945378,0.751252,238,361,238.0,0.0665
3,neutral,0.652842,0.713486,0.681818,1787,1953,1787.0,0.3599
4,admiration,0.503304,0.906746,0.647309,504,908,504.0,0.1673
5,remorse,0.432000,0.964286,0.596685,56,125,56.0,0.0230
6,fear,0.411765,0.897436,0.564516,78,170,78.0,0.0313
7,curiosity,0.380952,0.929577,0.540430,284,693,284.0,0.1277
8,surprise,0.355705,0.751773,0.482916,141,298,141.0,0.0549
9,desire,0.321951,0.795181,0.458333,83,205,83.0,0.0378


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
18,embarrassment,0.240964,0.540541,0.333333,37,83,37.0,0.0153
19,annoyance,0.210944,0.746875,0.328975,320,1133,320.0,0.2088
20,grief,0.192308,0.833333,0.312500,6,26,6.0,0.0048
21,confusion,0.187845,0.888889,0.310148,153,724,153.0,0.1334
22,pride,0.218750,0.437500,0.291667,16,32,16.0,0.0059
23,excitement,0.181598,0.728155,0.290698,103,413,103.0,0.0761
24,disappointment,0.161342,0.668874,0.259974,151,626,151.0,0.1153
25,nervousness,0.150000,0.652174,0.243902,23,100,23.0,0.0184
26,relief,0.123077,0.727273,0.210526,11,65,11.0,0.0120
27,realization,0.127869,0.537931,0.206623,145,610,145.0,0.1124



TEST @0.5 | Macro-F1=0.4367 | Micro-F1=0.4890
Done.


In [3]:
# =========================
# GoEmotions EXP3: RoBERTa + Emoji Priors (learned from TRAIN)
# =========================

import random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support

# emoji lib
try:
    import emoji
except Exception:
    import sys
    !{sys.executable} -m pip -q install emoji
    import emoji

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "roberta-base"
MAX_LEN = 128
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 4
THRESH = 0.5

EMOJI_MIN_COUNT = 10
EMOJI_MAX = 300
EMOJI_PRIOR_SCALE = 0.35

print("Loading GoEmotions (simplified)...")
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
emotion_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(emotion_names)
print(f"Sizes: train={len(ds['train'])}, val={len(ds['validation'])}, test={len(ds['test'])}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def extract_emojis_from_text(text: str):
    return [ch for ch in text if ch in emoji.EMOJI_DATA]

def build_emoji_priors_from_train(train_split, num_labels=28, min_count=10, max_emojis=300):
    emoji_counts = {}
    emoji_label_counts = {}
    for ex in train_split:
        text = ex["text"]
        labs = ex["labels"]
        ems = extract_emojis_from_text(text)
        if not ems:
            continue
        for em in ems:
            emoji_counts[em] = emoji_counts.get(em, 0) + 1
            if em not in emoji_label_counts:
                emoji_label_counts[em] = np.zeros(num_labels, dtype=np.float64)
            for lab in labs:
                emoji_label_counts[em][lab] += 1.0

    filtered = [(em, c) for em, c in emoji_counts.items() if c >= min_count]
    filtered.sort(key=lambda x: x[1], reverse=True)
    filtered = filtered[:max_emojis]
    if len(filtered) == 0:
        print("⚠️ No frequent emojis found. Disabling emoji priors.")
        return [], None

    emoji_list = [em for em, _ in filtered]
    prior = torch.zeros(len(emoji_list), num_labels, dtype=torch.float32)

    for i, em in enumerate(emoji_list):
        counts = emoji_label_counts.get(em, np.zeros(num_labels, dtype=np.float64))
        sm = counts + 1.0
        sm = sm / sm.sum()
        prior[i] = torch.tensor(sm, dtype=torch.float32)

    print(f"✓ Learned emoji priors from TRAIN: {len(emoji_list)} emojis (min_count={min_count}, max={max_emojis})")
    return emoji_list, prior

emoji_list, emoji_prior = build_emoji_priors_from_train(
    ds["train"], num_labels=NUM_LABELS, min_count=EMOJI_MIN_COUNT, max_emojis=EMOJI_MAX
)

class GoEmotionsEmojiDataset(Dataset):
    def __init__(self, hf_split, tokenizer, emoji_list, max_len=128, num_labels=28):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.num_labels = num_labels
        self.emoji_list = emoji_list
        self.emoji_to_idx = {e:i for i,e in enumerate(emoji_list)}

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        y = torch.zeros(self.num_labels, dtype=torch.float32)
        for lab in item["labels"]:
            y[lab] = 1.0

        evec = torch.zeros(len(self.emoji_list), dtype=torch.float32)
        if self.emoji_list:
            for ch in extract_emojis_from_text(text):
                j = self.emoji_to_idx.get(ch, None)
                if j is not None:
                    evec[j] = 1.0

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": y,
            "emoji_vec": evec,
        }

class RobertaWithEmojiPriors(nn.Module):
    def __init__(self, model_name, num_labels, emoji_prior_matrix=None, prior_scale=0.35):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_labels)

        self.prior_scale = prior_scale
        if emoji_prior_matrix is not None:
            self.register_buffer("emoji_prior", emoji_prior_matrix)  # [E, L]
        else:
            self.emoji_prior = None

    def forward(self, input_ids, attention_mask, emoji_vec=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)

        if (self.emoji_prior is not None) and (emoji_vec is not None) and (emoji_vec.size(1) == self.emoji_prior.size(0)):
            bias_raw = emoji_vec @ self.emoji_prior
            denom = emoji_vec.sum(dim=1, keepdim=True).clamp(min=1.0)
            bias = bias_raw / denom
            logits = logits + (self.prior_scale * bias)

        return logits

def evaluate(model, loader, device, thresh=0.5, return_arrays=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            evec = batch["emoji_vec"].to(device)
            y = batch["labels"].cpu().numpy()

            logits = model(ids, mask, emoji_vec=evec)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs); all_labels.append(y)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    preds = (all_probs > thresh).astype(int)

    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    micro = f1_score(all_labels, preds, average="micro", zero_division=0)

    if return_arrays:
        return all_probs, all_labels, macro, micro
    return macro, micro

def emotion_level_report(probs, labels, emotion_names, threshold=0.5, sort_by="f1"):
    preds = (probs > threshold).astype(int)
    p, r, f1, support = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    df = pd.DataFrame({"emotion": emotion_names, "precision": p, "recall": r, "f1": f1, "support": support})
    df["pred_count"] = preds.sum(axis=0)
    df["true_count"] = labels.sum(axis=0)
    df["pred_rate"] = (df["pred_count"] / max(len(preds), 1)).round(4)
    return df.sort_values(sort_by, ascending=False).reset_index(drop=True)

train_ds = GoEmotionsEmojiDataset(ds["train"], tokenizer, emoji_list, MAX_LEN, NUM_LABELS)
val_ds   = GoEmotionsEmojiDataset(ds["validation"], tokenizer, emoji_list, MAX_LEN, NUM_LABELS)
test_ds  = GoEmotionsEmojiDataset(ds["test"], tokenizer, emoji_list, MAX_LEN, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False)

model = RobertaWithEmojiPriors(
    MODEL_NAME, NUM_LABELS,
    emoji_prior_matrix=emoji_prior,
    prior_scale=EMOJI_PRIOR_SCALE
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val = -1.0
best_path = "roberta_exp3_emoji_priors_best.pt"

print("\nTraining EXP3...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        evec = batch["emoji_vec"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, mask, emoji_vec=evec)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_macro, val_micro = evaluate(model, val_loader, DEVICE, THRESH, return_arrays=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | val_macro={val_macro:.4f} | val_micro={val_micro:.4f}")

    if val_macro > best_val:
        best_val = val_macro
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ saved best (val_macro={best_val:.4f})")

print("\nTesting best checkpoint...")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_probs, test_labels, test_macro, test_micro = evaluate(model, test_loader, DEVICE, THRESH, return_arrays=True)

per_class_df = emotion_level_report(test_probs, test_labels, emotion_names, threshold=THRESH, sort_by="f1")
display(per_class_df.head(10))
display(per_class_df.tail(10))

print(f"\nTEST @0.5 | Macro-F1={test_macro:.4f} | Micro-F1={test_micro:.4f}")
print("Done.")


Loading GoEmotions (simplified)...
Sizes: train=43410, val=5426, test=5427
✓ Learned emoji priors from TRAIN: 36 emojis (min_count=10, max=300)

Training EXP3...
Epoch 1/4 | loss=0.1251 | val_macro=0.3616 | val_micro=0.5185
  ✓ saved best (val_macro=0.3616)
Epoch 2/4 | loss=0.0814 | val_macro=0.4581 | val_micro=0.5908
  ✓ saved best (val_macro=0.4581)
Epoch 3/4 | loss=0.0687 | val_macro=0.4939 | val_micro=0.5904
  ✓ saved best (val_macro=0.4939)
Epoch 4/4 | loss=0.0561 | val_macro=0.5113 | val_micro=0.5885
  ✓ saved best (val_macro=0.5113)

Testing best checkpoint...


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
0,gratitude,0.938596,0.911932,0.925072,352,342,352.0,0.0630
1,amusement,0.775920,0.878788,0.824156,264,299,264.0,0.0551
2,love,0.773234,0.873950,0.820513,238,269,238.0,0.0496
3,fear,0.730769,0.730769,0.730769,78,78,78.0,0.0144
4,admiration,0.680891,0.728175,0.703739,504,539,504.0,0.0993
5,remorse,0.602941,0.732143,0.661290,56,68,56.0,0.0125
6,joy,0.678322,0.602484,0.638158,161,143,161.0,0.0263
7,neutral,0.729228,0.550084,0.627113,1787,1348,1787.0,0.2484
8,sadness,0.633094,0.564103,0.596610,156,139,156.0,0.0256
9,optimism,0.618421,0.505376,0.556213,186,152,186.0,0.0280


,emotion,precision,recall,f1,support,pred_count,true_count,pred_rate
18,nervousness,0.562500,0.391304,0.461538,23,16,23.0,0.0029
19,excitement,0.519481,0.388350,0.444444,103,77,103.0,0.0142
20,approval,0.501818,0.393162,0.440895,351,275,351.0,0.0507
21,caring,0.462264,0.362963,0.406639,135,106,135.0,0.0195
22,disapproval,0.432692,0.337079,0.378947,267,208,267.0,0.0383
23,annoyance,0.455882,0.290625,0.354962,320,204,320.0,0.0376
24,disappointment,0.472222,0.225166,0.304933,151,72,151.0,0.0133
25,realization,0.455882,0.213793,0.291080,145,68,145.0,0.0125
26,relief,0.333333,0.181818,0.235294,11,6,11.0,0.0011
27,grief,0.000000,0.000000,0.000000,6,1,6.0,0.0002



TEST @0.5 | Macro-F1=0.5171 | Micro-F1=0.5964
Done.
